# Notebook 4 - Duplicate Data Handling

In this notebook I learn what duplicate records are, how to find exact and partial duplicates, how to detect them using selected columns, how to handle them with business rules, and why deleting duplicates without thinking can be dangerous.

**Dataset:** Telco Customer Churn (7043 rows, 21 columns).

**Problem with this dataset:** The real Telco data has no duplicate rows and `customerID` is unique (Sprint 4 and Notebook 1). So I cannot practise on the real data only. I do two things. (1) I create a **demo dataset** `df_dup`, where I inject duplicates of three known kinds into a copy, so I know the right answer and I can check my methods. (2) I run the final duplicate check on the real data. Everything that is demo data is labelled.

**Rule:** `df` (raw data) is never changed.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style("whitegrid")

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")   # raw data, never modified

def fix_data_types(data):
    # the function from Notebook 2
    out = data.copy()
    out["TotalCharges"] = pd.to_numeric(out["TotalCharges"], errors="coerce")
    out["SeniorCitizen"] = out["SeniorCitizen"].astype(bool)
    for col in ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]:
        out[col] = out[col].map({"Yes": True, "No": False})
    for col in ["gender", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
                "Contract", "PaymentMethod", "Churn"]:
        out[col] = out[col].astype("category")
    return out

def handle_missing_values(data):
    # the function from Notebook 3
    out = data.copy()
    new_customer = out["tenure"] == 0
    out.loc[new_customer, "TotalCharges"] = out.loc[new_customer, "TotalCharges"].fillna(0.0)
    return out

df_filled = handle_missing_values(fix_data_types(df))
print("Shape after Notebook 2 and 3:", df_filled.shape, "| missing values:", int(df_filled.isna().sum().sum()))

Shape after Notebook 2 and 3: (7043, 21) | missing values: 0


## 1. Problems from Sprint 4 Related to Duplicates

| # | Problem | Why it exists | Planned fix |
|---|---|---|---|
| 1 | No exact duplicate rows were found | the data has one row per customer | keep a duplicate check in the final workflow |
| 2 | `customerID` is unique | it is the key of the table | use it as the business key |
| 3 | Rows can look the same on the feature columns and still be different customers | many customers have the same plan | do not remove rows by features only (checked in Section 5) |
| 4 | Duplicates can make the model score too high | the same row can land in both train and test | show the effect in Section 9 |

## 2. What are Duplicate Records?

**Concept:** A duplicate record is a row that describes the same real-world thing as another row. Duplicates come from double data entry, a file loaded twice, joins that repeat rows, a system that saves a record again after an update, or different spellings of the same ID. They make some entities count more than once, so they change totals, averages and class balance, and they can copy a row into both the train and the test set.

There are two main kinds:
- **Exact duplicates:** all the column values are identical.
- **Partial duplicates:** the key (for example `customerID`) is the same, but some other values are different. They can also be rows that look different only because of formatting (spaces, upper or lower case).

**Examples:**
- Simple: the same row copied twice in a spreadsheet.
- Real-world: a person registered twice with "john" and "John " as the name.
- Business: a customer record saved again after the monthly bill was updated.
- AI/ML: the same row in train and test makes the test score look better than it is.

**Implementation:** The real data has no duplicates, so I create the demo dataset `df_dup` from the clean data (after Notebooks 2 and 3), with three kinds of duplicates, and I keep the ground truth.

| Kind | How many | How I created it |
|---|---|---|
| Exact duplicates | 60 | copies of 60 random rows |
| Updated records (partial) | 30 | a copy of a row with `tenure + 1` and a bigger `TotalCharges` (a later snapshot of the same customer) |
| Hidden duplicates (formatting) | 20 | a copy with the `customerID` in lower case and a space at the end |

In [2]:
# DEMO data: injected duplicates, ground truth is known
rng = np.random.default_rng(11)
pool = rng.permutation(len(df_filled))
exact_idx, updated_idx, hidden_idx = pool[:60], pool[60:90], pool[90:110]      # no row is used twice

exact_copies = df_filled.iloc[exact_idx].copy()

updated = df_filled.iloc[updated_idx].copy()
updated["tenure"] = updated["tenure"] + 1
updated["TotalCharges"] = (updated["TotalCharges"] + updated["MonthlyCharges"]).round(2)

hidden = df_filled.iloc[hidden_idx].copy()
hidden["customerID"] = hidden["customerID"].str.lower() + " "

df_dup = pd.concat([df_filled, exact_copies, updated, hidden], ignore_index=True)

updated_ids = df_filled.iloc[updated_idx]["customerID"].tolist()        # ground truth for later checks

print("Real data :", df_filled.shape)
print("Demo data :", df_dup.shape, "(60 + 30 + 20 = 110 extra rows)")
print("\nReal data -> exact duplicates:", int(df_filled.duplicated().sum()), "| duplicate customerID:", int(df_filled["customerID"].duplicated().sum()))
print("Demo data -> exact duplicates:", int(df_dup.duplicated().sum()), "| duplicate customerID:", int(df_dup["customerID"].duplicated().sum()))

Real data : (7043, 21)
Demo data : (7153, 21) (60 + 30 + 20 = 110 extra rows)

Real data -> exact duplicates: 0 | duplicate customerID: 0
Demo data -> exact duplicates: 60 | duplicate customerID: 90


**What the code does:**
- It takes a random permutation of the row numbers and uses separate parts of it for the three kinds, so a row is never used for two kinds.
- Exact duplicates: it copies 60 rows without any change.
- Updated records: it copies 30 rows and changes them like a later snapshot of the same customer: one more month of `tenure`, and `TotalCharges` increased by one monthly charge.
- Hidden duplicates: it copies 20 rows and changes the `customerID` to lower case with a space at the end, as it could happen with manual entry.
- `pd.concat` puts the real rows and the 110 injected rows together. `updated_ids` stores the IDs of the updated customers, so I can check my result later.
- It prints the number of exact duplicates and duplicate IDs in the real data and in the demo data.

**Why this method was used:** On the real data the number of duplicates is 0, so I cannot see whether my code works. If I add duplicates that I created myself, I know exactly how many there are (60, 30 and 20) and I can check my detection against this number. I chose three kinds, because each one needs a different detection and a different decision. A fixed seed (`default_rng(11)`) makes the demo repeatable.

**Before:** The real data has 7043 rows, 0 exact duplicates and 0 duplicate IDs.

**After:** The demo data has 7153 rows. The check finds 60 exact duplicates and 90 duplicate IDs (60 exact copies and 30 updated records). The 20 hidden duplicates are not found by either check, because their ID text is different.

**Effect on ML:** The demo shows that a simple check on the whole row misses two of the three kinds of duplicates. I need several kinds of checks.

## 3. Exact Duplicates: `duplicated()` and `drop_duplicates()`

**Concept:** `duplicated()` returns True for every row that repeats an earlier row, and `drop_duplicates()` removes them. The `keep` option decides what is marked: `keep="first"` keeps the first one (marks the later ones), `keep="last"` keeps the last one, and `keep=False` marks all the rows of a duplicate group, so I can look at all of them.

- **When to use:** all the values are identical, and the rows describe the same entity (a copy by mistake).
- **When not to use:** the table has no unique key, and identical rows can be real separate events (two identical purchases).
- **Advantages:** simple, safe when the rows are true copies, and it removes the risk of a repeated row.
- **Limitations:** finds only the copies that are identical in every column, so it misses partial and formatting duplicates.

**Examples:**
- Simple: the same row pasted twice.
- Real-world: a CSV file appended twice to a database.
- Business: a customer file loaded two times by the same job.
- AI/ML: a repeated training row gives the model a double weight.

**Implementation:**

In [3]:
print("Exact duplicates with keep='first':", int(df_dup.duplicated(keep="first").sum()))
print("Exact duplicates with keep='last' :", int(df_dup.duplicated(keep="last").sum()))
print("Rows involved with keep=False     :", int(df_dup.duplicated(keep=False).sum()), "(every row of every duplicate group)")
print("Share of the demo data (%)        :", round(df_dup.duplicated().mean() * 100, 2))

# look at a few groups: both copies together
groups = df_dup[df_dup.duplicated(keep=False)].sort_values("customerID")
print()
print(groups[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract"]].head(6))

deduped = df_dup.drop_duplicates()
print("\nShape before:", df_dup.shape, "| after drop_duplicates():", deduped.shape, "| removed:", len(df_dup) - len(deduped))

Exact duplicates with keep='first': 60
Exact duplicates with keep='last' : 60
Rows involved with keep=False     : 120 (every row of every duplicate group)
Share of the demo data (%)        : 0.84

      customerID  tenure  MonthlyCharges  TotalCharges        Contract
6020  0020-JDNXP      34           61.25       1993.20        One year
7100  0020-JDNXP      34           61.25       1993.20        One year
1241  0096-FCPUF      30           64.50       1888.45  Month-to-month
7093  0096-FCPUF      30           64.50       1888.45  Month-to-month
1797  0839-QNXME      30           19.40        578.50        Two year
7046  0839-QNXME      30           19.40        578.50        Two year

Shape before: (7153, 21) | after drop_duplicates(): (7093, 21) | removed: 60


**What the code does:**
- `duplicated(keep="first")` marks every row that already appeared before, and the sum counts them. `keep="last"` marks the earlier ones instead, and the count is the same.
- `duplicated(keep=False)` marks all the rows of every duplicate group (the original and the copy), so the count is double.
- It calculates the share of duplicate rows with `.mean()` on the True/False column.
- It selects the rows of the groups and sorts them by `customerID`, so each original and its copy are next to each other, and prints a few of them.
- `drop_duplicates()` removes the repeated rows, and the code prints the shape before and after.

**Why this method was used:** I look at the duplicates before removing them. `keep=False` is the best option for looking, because it shows both copies and I can see that they are really identical. `keep="first"` is used for the removal, because for exact copies it does not matter which one stays. The counts with `first` and `last` must be the same, and this is a quick check that the duplicates are real groups. The share (%) tells how big the problem is.

**Before:** 7153 rows, with 60 rows that repeat an earlier row.

**After:** 7093 rows. The 60 exact copies are gone, and the 30 updated records and the 20 hidden duplicates are still there.

**Why appropriate:** For rows that are identical in all 21 columns, including the unique ID, there is no information in the second copy, so removing it is safe.

**Effect on ML:** The exact copies are removed and no row has a double weight. But I know that the data still has other duplicates, which this method cannot see.

## 4. Partial Duplicates

**Concept:** Partial duplicates are rows that describe the same entity but are not identical. I find them by choosing the key columns that identify the entity (`customerID`) and looking for repeated keys with `duplicated(subset=...)`. After this, I compare the rows of the group to see which columns are different. There is also a kind of partial duplicate where the key looks different only because of formatting (case, spaces), and I find them after cleaning the text.

- **When to use:** a business key exists, and the same key must appear only once.
- **When not to use:** the key is not unique by design (for example one customer with many orders).
- **Advantages:** finds duplicates that the full row check misses.
- **Limitations:** needs a good key, and the decision between the versions needs a business rule.

**Examples:**
- Simple: two rows with the same ID and a different phone number.
- Real-world: a person who moved and has an old and a new address record.
- Business: a customer with two snapshots of the bill from two different months.
- AI/ML: the model sees the same customer twice, as if there were two people.

**Implementation:**

In [4]:
# step 1: repeated key after the exact copies are removed
partial = deduped[deduped.duplicated(subset="customerID", keep=False)].sort_values(["customerID", "tenure"])
print("Rows that share a customerID after removing exact copies:", len(partial), "| customers:", partial["customerID"].nunique())
print()
print(partial[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract"]].head(6))

# step 2: which columns are different inside the groups?
different = partial.groupby("customerID").nunique(dropna=False).max()
print("\nColumns that differ inside the groups:", different[different > 1].index.tolist())

# step 3: hidden duplicates - the key looks different only because of formatting
key_clean = df_dup["customerID"].str.strip().str.upper()
print("\nDuplicate keys in the raw text      :", int(df_dup["customerID"].duplicated().sum()))
print("Duplicate keys after strip() + upper():", int(key_clean.duplicated().sum()))

Rows that share a customerID after removing exact copies: 60 | customers: 30

      customerID  tenure  MonthlyCharges  TotalCharges        Contract
5804  1555-HAPSU      71           23.90       1663.50        Two year
7104  1555-HAPSU      72           23.90       1687.40        Two year
2688  1569-TTNYJ       5           78.75        412.10  Month-to-month
7121  1569-TTNYJ       6           78.75        490.85  Month-to-month
5579  2225-ZRGSG      70           93.90       6579.05        Two year
7116  2225-ZRGSG      71           93.90       6672.95        Two year

Columns that differ inside the groups: ['tenure', 'TotalCharges']

Duplicate keys in the raw text      : 90
Duplicate keys after strip() + upper(): 110


**What the code does:**
- Step 1: `duplicated(subset="customerID", keep=False)` marks all the rows whose `customerID` appears more than once. I use it on `deduped` (exact copies already removed), so only the partial duplicates are left. I sort by ID and tenure to see the pairs, and print the number of rows and customers.
- Step 2: `groupby("customerID").nunique()` counts the different values of every column inside each group, and `.max()` takes the highest count per column. A column with a count above 1 is a column where the versions differ.
- Step 3: it cleans the key with `str.strip()` (removes spaces) and `str.upper()` (same case), and counts the duplicate keys again, before and after the cleaning.

**Why this method was used:** The question for a partial duplicate is not only "is it a duplicate", but also "what is different". The `nunique` per group answers it for all columns at once, without looking at each pair by eye. The cleaning of the key is needed because text matches are exact: `"abc "` and `"ABC"` are different for pandas. If I only used the raw text, I would not find these duplicates.

**Output meaning:** After removing the exact copies there are 30 customers with two rows each. The columns that differ are only `tenure` and `TotalCharges`, so they are two snapshots of the same customer in different months. After the cleaning of the key, the number of duplicate keys increases by 20, because the hidden duplicates now match.

**Effect on ML:** Each customer must be one row. If the two snapshots stay in the data, the model sees one person twice, with different values, and the same customer can appear in train and in test.

## 5. Duplicate Detection Based on Selected Columns

**Concept:** The `subset` option decides which columns are compared. The choice of columns changes the meaning of "duplicate":
- all columns: the rows are copies,
- the business key (ID): the rows are the same entity,
- only the feature columns: the rows look the same, but they may be different entities,
- very few columns: almost every row is a duplicate and the result is meaningless.

**Examples:**
- Simple: two rows with the same name are not always the same person.
- Real-world: two people with the same date of birth and city.
- Business: many customers have the same plan and payment method.
- AI/ML: if the features are identical but the target is different, the model gets a conflict.

**Implementation:** I test different subsets on the **real** data (after Notebooks 2 and 3).

In [5]:
feature_cols = [c for c in df_filled.columns if c != "customerID"]

subsets = {
    "all columns": None,
    "customerID only": ["customerID"],
    "all columns except customerID": feature_cols,
    "Contract + InternetService + PaymentMethod": ["Contract", "InternetService", "PaymentMethod"],
    "gender + Churn": ["gender", "Churn"],
}

rows = []
for name, cols in subsets.items():
    flagged = int(df_filled.duplicated(subset=cols).sum())
    rows.append((name, flagged, round(flagged / len(df_filled) * 100, 2)))

pd.DataFrame(rows, columns=["compared columns", "rows flagged as duplicate", "share_%"])

,compared columns,rows flagged as duplicate,share_%
0,all columns,0,0.00
1,customerID only,0,0.00
2,all columns except customerID,22,0.31
3,Contract + InternetService + PaymentMethod,7007,99.49
4,gender + Churn,7039,99.94


In [6]:
# customers that are identical on every feature but have a different customerID
same_features = df_filled[df_filled.duplicated(subset=feature_cols, keep=False)].sort_values(feature_cols)
print("Rows identical on all features (different customerID):", len(same_features))
if len(same_features) > 0:
    print("Different customerID inside those rows:", same_features["customerID"].nunique())
    print()
    print(same_features[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]].head(6))

Rows identical on all features (different customerID): 42
Different customerID inside those rows: 42

      customerID  tenure  MonthlyCharges  TotalCharges        Contract Churn
6491  9728-FTTVZ       1           69.20         69.20  Month-to-month   Yes
6764  7660-HDPJV       1           69.20         69.20  Month-to-month   Yes
4495  4702-IOQDC       1           70.10         70.10  Month-to-month   Yes
6267  0328-GRPMV       1           70.10         70.10  Month-to-month   Yes
5522  2619-WFQWU       1           70.15         70.15  Month-to-month   Yes
5759  9985-MWVIX       1           70.15         70.15  Month-to-month   Yes


**What the code does:**
- It defines five different `subset` choices, from all columns to only two columns, and counts how many rows `duplicated(subset=...)` flags in each case on the real data.
- In the second cell it takes the rows that are identical on all feature columns and have different IDs, and prints them if there are any.

**Why this method was used:** The same function gives very different answers with different columns, so I test several options in a table before I decide. I use the real data, because the answer for it matters for the final decision. The second check is the important one for the review: if two rows have the same features but a different `customerID`, they are different customers, and the question is whether they should be removed.

**Output meaning:**
- All columns and `customerID` only: 0 rows. The real data has no duplicates by these definitions.
- Contract + InternetService + PaymentMethod and gender + Churn: almost every row is flagged. These are not real duplicates, only customers who share a plan.
- All columns except `customerID`: if some rows are flagged, they are different customers (different IDs) who happen to have the same features and the same total, which is possible for customers with similar plans.

**Decision:** I use the business key `customerID` and all columns together. I do not remove rows that are identical only on the features, because they have different IDs, so they are different customers.

**Effect on ML:** Removing customers with identical features would delete real customers and slightly change the data. Features that are identical with a different target would be a conflict for the model, but that is a question for the model, not for the removal of rows.

## 6. Handling Duplicate Records

**Concept:** There is more than one way to handle a duplicate. The choice depends on the kind of duplicate and on the business meaning.

| Strategy | What it does | Use when | Avoid when |
|---|---|---|---|
| Remove (keep first) | keeps the first row, drops the rest | exact copies | the versions are different |
| Keep the latest by a rule | sorts by a column (date, version, tenure) and keeps the newest | updated records | there is no column that shows which is newer |
| Merge / aggregate | combines the versions into one row (max, sum, mean, first) | each version has part of the information | the combination has no meaning |
| Flag, do not remove | adds a column `is_duplicate` | I am not sure, or a person must check | the flag is forgotten, and duplicates stay in the model |
| Manual review | a person decides | high-value or high-risk records | very large data |

**Examples:**
- Simple: keep the row with the latest date.
- Real-world: merge two contact records into one with the phone number from one and the email from the other.
- Business: keep the latest bill of each customer.
- AI/ML: one row per entity, so the model does not count a customer twice.

**Implementation:** I apply the strategies to the 30 updated customers of the demo data.

In [7]:
one_id = updated_ids[0]
print("Customer", one_id, "- the two versions in the demo data:")
print(deduped[deduped["customerID"] == one_id][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]])

# strategy A: keep the first row (depends only on the file order)
keep_first = deduped.drop_duplicates(subset="customerID", keep="first")

# strategy B: keep the latest version by a business rule (highest tenure = latest snapshot)
keep_latest = deduped.sort_values("tenure", kind="stable").drop_duplicates(subset="customerID", keep="last").sort_index()

# strategy C: merge the versions into one row
merged = (deduped[deduped["customerID"].isin(updated_ids)]
          .groupby("customerID").agg(tenure=("tenure", "max"), TotalCharges=("TotalCharges", "max"), MonthlyCharges=("MonthlyCharges", "first")))

# strategy D: flag instead of removing
flagged = deduped.assign(is_duplicate_key=deduped.duplicated(subset="customerID", keep=False))

print("\nA keep first :", keep_first.loc[keep_first["customerID"] == one_id, ["tenure", "TotalCharges"]].values.tolist())
print("B keep latest:", keep_latest.loc[keep_latest["customerID"] == one_id, ["tenure", "TotalCharges"]].values.tolist())
print("C merged     :", merged.loc[one_id, ["tenure", "TotalCharges"]].tolist())
print("D flagged rows:", int(flagged["is_duplicate_key"].sum()), "(rows stay in the data, with a flag)")

Customer 6303-KFWSL - the two versions in the demo data:
      customerID  tenure  MonthlyCharges  TotalCharges
1712  6303-KFWSL      29           55.25       1620.20
7103  6303-KFWSL      30           55.25       1675.45

A keep first : [[29.0, 1620.2]]
B keep latest: [[30.0, 1675.45]]
C merged     : [30.0, 1675.45]
D flagged rows: 60 (rows stay in the data, with a flag)


**What the code does:**
- It prints the two versions of one updated customer, so I can see what is different.
- Strategy A: `drop_duplicates(subset="customerID", keep="first")` keeps the row that comes first in the file.
- Strategy B: it sorts the rows by `tenure` (a stable sort) and keeps the last row of each ID, that is, the one with the highest tenure. `sort_index()` brings the original order back.
- Strategy C: for the updated customers it groups by ID and builds one row with the highest tenure, the highest `TotalCharges` and the first `MonthlyCharges`.
- Strategy D: it only adds a column `is_duplicate_key` and does not remove anything.
- It prints the result of each strategy for the same customer.

**Why this method was used:** The same duplicates can be solved in four ways, and I want to see the results side by side. For strategy B I need a column that tells which version is newer. In this dataset `tenure` plays that role, because a later snapshot of the same customer has more months. `kind="stable"` keeps the original order for rows with the same value, so the result is repeatable.

**Output meaning:** In this demo the file contains the original row first and the update later, so strategy A keeps the old version (stale data), while B and C keep the newer one. The result of A is correct only by chance, and it is the wrong version here.

**Effect on ML:** The strategy decides which values the model learns from. A wrong choice puts old values into the training data. The strategy must be a business decision with a reason, and not the default of a function.

## 7. Business Rules for Duplicate Removal

**Concept:** A business rule is a written decision about what a duplicate is and what happens to it. Without a rule, the result of `drop_duplicates()` depends on the order of the rows. My rules for this dataset:

| # | Rule | Reason |
|---|---|---|
| 1 | The business key is `customerID`, compared after `strip()` and `upper()` | the same customer can be typed in different ways |
| 2 | Rows that are identical in all columns are copies, so keep one | no information in the second copy |
| 3 | If the same customer has several versions, keep the one with the highest `tenure` | it is the latest snapshot |
| 4 | Never remove rows because the features are equal | different IDs are different customers |
| 5 | Every removed row is saved in an audit table with the reason | the decision can be checked and undone |

**Examples:**
- Simple: "if the ID is the same, keep the newest row".
- Real-world: a bank keeps the latest address of a client.
- Business: the audit table is shown to the data owner.
- AI/ML: the same function runs on the training data and in production, so both are treated in the same way.

**Implementation:** I write the rules as a function that returns the clean data and the audit table.

In [8]:
def handle_duplicates(data, key="customerID", order_col="tenure"):
    out = data.copy()                                                   # raw data is never changed
    audit = []

    # rule 1: the same key written in different ways
    out[key] = out[key].astype(str).str.strip().str.upper()

    # rule 2: exact copies, keep the first
    exact = out.duplicated(keep="first")
    audit.append(out[exact].assign(reason="exact duplicate"))
    out = out[~exact]

    # rule 3: several versions of the same key, keep the highest order_col
    older = out.sort_values(order_col, kind="stable").duplicated(subset=key, keep="last").reindex(out.index)
    audit.append(out[older].assign(reason="older version of the same " + key))
    out = out[~older]

    return out, pd.concat(audit)

# run on the DEMO data and check against the ground truth
demo_clean, demo_audit = handle_duplicates(df_dup)

print("Demo rows before:", len(df_dup), "| after:", len(demo_clean), "| removed:", len(demo_audit))
print(demo_audit["reason"].value_counts().to_dict())
print("Key is unique now:", demo_clean["customerID"].is_unique)
print("Same size as the original real data:", len(demo_clean) == len(df_filled))

# the updated customers must have the NEW tenure (original + 1)
original_tenure = df_filled.set_index("customerID").loc[updated_ids, "tenure"]
kept_tenure = demo_clean.set_index("customerID").loc[updated_ids, "tenure"]
print("Updated customers that kept the newest version:", int((kept_tenure.values == original_tenure.values + 1).sum()), "of", len(updated_ids))

Demo rows before: 7153 | after: 7043 | removed: 110
{'exact duplicate': 80, 'older version of the same customerID': 30}
Key is unique now: True
Same size as the original real data: True
Updated customers that kept the newest version: 30 of 30


**What the code does:**
- The function `handle_duplicates` makes a copy of the data and applies the rules in the right order.
- Rule 1: it cleans the key column with `strip()` and `upper()`.
- Rule 2: `duplicated(keep="first")` finds the exact copies. They go to the audit list with the reason, and the rest stays.
- Rule 3: it sorts by the order column, finds the repeated keys, and marks all but the last (highest value) as older versions. They also go to the audit list, and the newest stay. `reindex(out.index)` puts the True/False marks back in the original row order.
- It returns the clean data and one audit table with the reason for every removed row.
- Then it runs on the demo data and checks the result: the number of removed rows by reason, the uniqueness of the key, the size, and whether all 30 updated customers kept the newest version.

**Why this method was used:** The order of the steps matters. The key must be cleaned first, otherwise the hidden duplicates are not found. The exact copies are removed second, because they are the safest. The version rule comes last, because it needs a decision. The audit table is important: it gives a reason for each removed row, so I can show it in the review and I can undo a wrong decision. I checked the function with the demo data, because for it I know the right answer: I expect 80 exact copies (60 plus the 20 hidden), 30 older versions and 7043 rows.

**Output meaning:** The result has the same number of rows as the original real data, the key is unique, the audit table has 80 exact duplicates and 30 older versions, and all 30 updated customers kept their newest version.

**Effect on ML:** There is exactly one row per customer, with the latest values. The audit table is a record of what was changed, which is needed for the documentation of the sprint.

## 8. Why Blindly Deleting Duplicates Can Be Dangerous

**Concept:** `drop_duplicates()` always works and never gives an error, even when the result is wrong. It removes rows that are equal in the columns that I compare, and it does not know if they are real duplicates. Four situations where blind deletion does damage:

1. **Too few columns are compared:** valid rows are removed.
2. **Repeated rows are real events:** two identical purchases are two sales.
3. **The wrong version is kept:** the default is the first row, not the newest.
4. **Duplicates are removed from test data or after the split:** the test set changes and the numbers cannot be compared. (The duplicates must be handled before the split, see Section 9.)

**Examples:**
- Simple: two people called "John Smith" are not one person.
- Real-world: a customer who buys the same coffee twice a day.
- Business: the revenue of the month is lower after the "cleaning".
- AI/ML: the class balance and the number of rows change without anyone noticing.

**Implementation:**

In [9]:
# 1) too few columns: real Telco data
wrong_subset = ["Contract", "InternetService", "PaymentMethod"]
blind = df_filled.drop_duplicates(subset=wrong_subset)
print("1) TOO FEW COLUMNS")
print("   rows before:", len(df_filled), "| after drop_duplicates(subset=" + str(wrong_subset) + "):", len(blind))
print("   customers deleted:", len(df_filled) - len(blind), "(" + str(round((1 - len(blind) / len(df_filled)) * 100, 1)) + "%)")

# 2) repeated rows that are real events (DEMO orders table)
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer": ["A", "A", "B", "C", "C"],
    "item":     ["coffee", "coffee", "tea", "coffee", "coffee"],
    "date":     ["2024-05-01"] * 5,
    "amount":   [3.5, 3.5, 2.8, 3.5, 3.5],
})
blind_orders = orders.drop_duplicates(subset=["customer", "item", "date", "amount"])
print("\n2) REAL REPEATED EVENTS (demo orders)")
print("   orders before:", len(orders), "| revenue:", orders["amount"].sum())
print("   orders after blind removal:", len(blind_orders), "| revenue:", blind_orders["amount"].sum())
print("   safe check with the unique order_id, duplicates:", int(orders.duplicated(subset="order_id").sum()))

# 3) wrong version kept: the file order is not a business rule
shuffled = deduped.sample(frac=1, random_state=5)                       # the rows come in a different order
by_file_order = shuffled.drop_duplicates(subset="customerID", keep="first").set_index("customerID")
by_rule = shuffled.sort_values("tenure", kind="stable").drop_duplicates(subset="customerID", keep="last").set_index("customerID")
newest_tenure = df_filled.set_index("customerID").loc[updated_ids, "tenure"].values + 1
print("\n3) WRONG VERSION KEPT (rows in random file order)")
print("   updated customers with the newest version, keep='first':", int((by_file_order.loc[updated_ids, "tenure"].values == newest_tenure).sum()), "of", len(updated_ids))
print("   updated customers with the newest version, rule by tenure:", int((by_rule.loc[updated_ids, "tenure"].values == newest_tenure).sum()), "of", len(updated_ids))

1) TOO FEW COLUMNS
   rows before: 7043 | after drop_duplicates(subset=['Contract', 'InternetService', 'PaymentMethod']): 36
   customers deleted: 7007 (99.5%)

2) REAL REPEATED EVENTS (demo orders)
   orders before: 5 | revenue: 16.8
   orders after blind removal: 3 | revenue: 9.8
   safe check with the unique order_id, duplicates: 0

3) WRONG VERSION KEPT (rows in random file order)
   updated customers with the newest version, keep='first': 20 of 30
   updated customers with the newest version, rule by tenure: 30 of 30


**What the code does:**
- Case 1: it runs `drop_duplicates` on the real data with only three columns (contract, internet service, payment method), and counts how many customers are deleted.
- Case 2: it builds a small demo orders table with two customers who bought the same coffee twice on the same day. It removes duplicates by customer, item, date and amount, and compares the number of orders and the revenue. Then it checks with the unique `order_id` for comparison.
- Case 3: it shuffles the rows of the demo data, so the file order is random. It keeps the first row of each customer (default behaviour), and compares with the rule "highest tenure". For the 30 updated customers it counts how many kept the newest version.

**Why this method was used:** I show the danger with numbers, not with words. In case 1 the result is easy to see: the table collapses. Case 2 shows the same problem in a transaction table, where the damage is money. Case 3 is the most quiet one: nothing is deleted by mistake, but the wrong row stays. I shuffled the rows to show that the first row has no meaning, because in a real file the order can depend on how the data was loaded.

**Output meaning:**
- Case 1: almost all customers are deleted (only one row per combination of the three columns stays, so about 36 rows).
- Case 2: two real sales are lost and the revenue falls from 16.8 to 9.8. The `order_id` shows there are no real duplicates.
- Case 3: with `keep="first"` a part of the updated customers keep the old version (the exact number depends on the shuffle), but the rule by tenure keeps the newest for all 30.

**Effect on ML:** Blind removal can destroy the data, change totals and class balance, or keep old values, and none of this gives an error. So before every `drop_duplicates()` I ask three questions: which columns identify the entity, are repeated rows possible as real events, and which version must stay.

## 9. Why Duplicates Are a Problem for Machine Learning

**Concept:** If the same row exists twice and the data is split randomly, one copy can be in the training set and the other in the test set. The model is then tested on a row that it has already seen, and the test score is too high. This is a case of train-test contamination (data leakage, Notebook 13). Duplicates also give a higher weight to the repeated rows during training.

**Examples:**
- Simple: a student sees the exam question during the lesson.
- Real-world: the same medical image in the training and test set.
- Business: a churn model with a very good test score that does not work in production.
- AI/ML: a decision tree that memorises rows gives a high score on the duplicates.

**Implementation:** I use the real data, add 1500 exact copies of random rows (a heavy demo duplication of about 21%), split the data in two ways, and train an unrestricted decision tree (it can memorise rows, so the effect is easy to see). The features are made with `get_dummies` only for this experiment. The real encoding is in Notebook 7.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def make_xy(data):
    X = pd.get_dummies(data.drop(columns=["customerID", "Churn"])).astype(float)
    y = (data["Churn"] == "Yes").astype(int)
    return X, y

def run(data, label):
    X, y = make_xy(data)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    train_rows = set(pd.util.hash_pandas_object(X_tr, index=False))
    has_twin = pd.util.hash_pandas_object(X_te, index=False).isin(train_rows)
    model = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
    pred = model.predict(X_te)
    return {"setting": label, "rows": len(data), "test rows with an identical row in train_%": round(has_twin.mean() * 100, 1),
            "accuracy (all test rows)": round(accuracy_score(y_te, pred), 3),
            "accuracy (test rows without twin)": round(accuracy_score(y_te[~has_twin], pred[~has_twin.to_numpy()]), 3)}

heavy = pd.concat([df_filled, df_filled.sample(1500, random_state=1)], ignore_index=True)    # DEMO: 1500 exact copies
heavy_clean = heavy.drop_duplicates()                                                       # duplicates removed BEFORE the split

pd.DataFrame([run(heavy, "duplicates inside, then split"), run(heavy_clean, "duplicates removed, then split")]).set_index("setting")

,rows,test rows with an identical row in train_%,accuracy (all test rows),accuracy (test rows without twin)
setting,,,,
"duplicates inside, then split",8543,28.8,0.803,0.730
"duplicates removed, then split",7043,1.1,0.718,0.722


**What the code does:**
- `make_xy` builds the features (`get_dummies`, all numeric) and the target from a dataframe, without the ID.
- `run` splits the data randomly 80/20, calculates a hash of every row, and counts how many test rows have an identical row in the training set (a "twin"). It trains an unrestricted decision tree and calculates the accuracy for all the test rows and only for the test rows without a twin.
- The experiment runs twice: once on the data with 1500 copies, and once after `drop_duplicates()`, before the split.

**Why this method was used:** The decision tree has no depth limit, so it can remember its training rows, and this makes the effect of duplicates clear. The hash of a row is a quick way to compare the rows without a slow comparison. The accuracy for the rows without a twin is the honest score. I compare the same data with and without duplicates, so the only thing that changes is the duplicates.

**Output meaning:** In the data with duplicates, a large share of the test rows (more than a quarter in my test run) have an identical row in the training set, and the accuracy over all the test rows is clearly higher than the accuracy over the rows without a twin. After removing the duplicates, no test row has a twin, and the accuracy is lower, but it is the honest value. The exact numbers will be a little different in your run.

**Insight obtained:** The higher score with duplicates is not a better model, it is the same model that gets credit for rows that it has already seen.

**Effect on ML:** Duplicates must be handled before the split, and the rules must be applied to the whole data. This is also the reason to use the business key. After the cleaning, the same customer cannot be in train and test.

## 10. Final Duplicate Check on the Real Data

I apply the function to the real data. Because the real data has no duplicates, the expected result is that nothing is removed. This is a validation step, and the same step will be in the final workflow (Notebook 16).

In [11]:
df_dedup, audit = handle_duplicates(df_filled)

assert len(audit) == 0, "unexpected duplicates were found"
assert df_dedup["customerID"].is_unique, "customerID is not unique"
assert not df_dedup.duplicated().any(), "exact duplicates found"

summary = pd.DataFrame({
    "before": [len(df_filled), int(df_filled.duplicated().sum()), int(df_filled["customerID"].duplicated().sum())],
    "after":  [len(df_dedup),  int(df_dedup.duplicated().sum()),  int(df_dedup["customerID"].duplicated().sum())],
}, index=["rows", "exact duplicates", "duplicate customerID"])
print(summary)
print("\nRows removed:", len(audit), "| raw data untouched, rows in df:", len(df))

df_dedup.to_csv("telco_step4_duplicates_handled.csv", index=False)

if os.path.exists("data_quality_issue_log.csv"):
    log = pd.read_csv("data_quality_issue_log.csv")
    log.loc[log["id"] == 3, "status"] = "done (Notebook 4)"
    log.to_csv("data_quality_issue_log.csv", index=False)
    print(log[["id", "column", "problem", "status"]].head(3))
else:
    print("data_quality_issue_log.csv not found (run Notebook 1 first)")

                      before  after
rows                    7043   7043
exact duplicates           0      0
duplicate customerID       0      0

Rows removed: 0 | raw data untouched, rows in df: 7043
   id        column                   problem             status
0   1  TotalCharges            stored as text  done (Notebook 2)
1   2  TotalCharges  11 hidden missing values  done (Notebook 3)
2   3      all rows           duplicate check  done (Notebook 4)


**What the code does:**
- It runs `handle_duplicates` on the real data and receives the clean data and the audit table.
- Three `assert` statements stop the code if the audit table is not empty, if the ID is not unique, or if an exact duplicate exists.
- It prints a before/after table, saves the dataset, and updates the status of issue 3 in the issue log.

**Why this method was used:** On the real data the function is a safety check, not a cleaning step. The `assert` lines are cheap validations: if the data changes in the future (a new file with duplicates), the pipeline stops and tells me, instead of silently using bad data. The normalisation of the ID is also tested, because the real IDs are already in upper case without spaces, so nothing should change.

**Before and after:** 7043 rows, 0 exact duplicates, 0 duplicate IDs, and the same after. The audit table is empty.

**Effect on ML:** The data is ready for the next step, validation (Notebook 5). The same function is used in the final pipeline, and it protects the model against duplicates in new data.

## 11. Documentation of the Decisions

**Decision 1 - Real data: no duplicates, keep the check**

| Item | Documentation |
|---|---|
| Problem | I must check whether the data contains duplicate customers |
| Analysis | `customerID` is unique and no row is repeated. Rows that are identical on the features (if any) have different IDs, so they are different customers |
| Technique selected | `duplicated()` on all columns and on the key `customerID`, inside the function `handle_duplicates()` with `assert` checks |
| Reason | A check on the business key plus the full row is the safest definition. A check on the features only would delete real customers |
| Implementation | Sections 5 and 10 |
| Result | 0 rows removed, 7043 rows, the key is unique |
| Impact | No customer is counted twice, and no row can appear in both train and test |

**Decision 2 - Rules for duplicates in new data (tested on demo data)**

| Item | Documentation |
|---|---|
| Problem | New data can contain exact copies, several versions of one customer, and IDs with different formatting |
| Analysis | These come from reloaded files, updated records and manual entry. They are not visible with a single `drop_duplicates()` |
| Technique selected | Clean the key, remove the exact copies, keep the version with the highest `tenure`, and save every removed row with a reason |
| Reason | Each kind of duplicate needs its own rule. The order of the rows must not decide which version stays |
| Implementation | `handle_duplicates()` in Section 7 |
| Result | In the demo data all 110 injected duplicates are found, 7043 rows remain, and all 30 updated customers keep the newest version |
| Impact | Training data has one current row per customer, and the audit table makes the decision traceable |

# Summary

| Topic | Key idea |
|---|---|
| Duplicate records | rows that describe the same entity more than once |
| Exact duplicates | identical in all columns, `duplicated()` and `drop_duplicates()` |
| Partial duplicates | same key, different values, found with `subset` and `nunique` per group |
| Hidden duplicates | formatting differences, found after `strip()` and `upper()` |
| Selected columns | the `subset` decides what a duplicate is, so choose the business key |
| Handling | remove, keep the latest by a rule, merge, flag or review |
| Business rules | written, ordered rules, with an audit table |
| Blind deletion | too few columns, real repeated events and the wrong version are dangerous |
| ML impact | duplicates in train and test make the score too high |

**Key findings on the Telco data:**
- The real data has no exact duplicates and `customerID` is unique, so nothing is removed.
- Comparing only a few feature columns would flag almost every row, and these are different customers, not duplicates.
- In the demo data a plain `duplicated()` found only 60 of the 110 injected duplicates. The key check and the cleaning of the key found the rest.
- Duplicates in the data make the test accuracy look higher, because the model is tested on rows it has already seen.

**What I learned:** A duplicate is defined by the business meaning, not only by the function. `drop_duplicates()` is easy to run and hard to use correctly, so I must decide the key, the version and the order of the steps before I run it, and I must keep a record of what was removed.

**Next:** Notebook 5 - Data Validation.